## **LLM Quantization & RAG Pipeline with llama.cpp**


In [ ]:
!apt-get update && apt-get install -y cmake build-essential
!pip install -q mistral-common langchain langchain-community faiss-cpu sentence-transformers huggingface_hub llama-cpp-python

import os
if not os.path.exists('llama.cpp'):
    !git clone https://github.com/ggerganov/llama.cpp

%cd llama.cpp
!mkdir -p build
%cd build
!cmake ..
!make -j$(nproc)
%cd /content

In [ ]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    local_dir="/content/tinyllama-hf",
    local_dir_use_symlinks=False
)

!python3 /content/llama.cpp/convert_hf_to_gguf.py /content/tinyllama-hf \
    --outfile /content/tinyllama-1.1b-chat.gguf

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from llama_cpp import Llama

def prepare_sample_doc(doc_path="my_notes.txt"):
    if not os.path.exists(doc_path):
        with open(doc_path, "w") as f:
            f.write("""LLM Quantization reduces the precision of model weights (e.g., from FP16 to INT8 or INT4).
This process significantly lowers the memory footprint and accelerates inference on resource-constrained hardware.
GGUF is a popular file format designed for rapid loading and execution using libraries like llama.cpp.""")
        print(f"Sample document created at: {doc_path}")

def load_and_chunk_documents(file_path, chunk_size=500, chunk_overlap=100):
    loader = TextLoader(file_path)
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(documents)

def create_or_load_vectorstore(chunks, embedding_model, index_path="faiss_index"):
    if os.path.exists(index_path):
        print("Loading existing FAISS index...")
        return FAISS.load_local(index_path, embedding_model, allow_dangerous_deserialization=True)
    print("Creating new FAISS index...")
    db = FAISS.from_documents(chunks, embedding_model)
    db.save_local(index_path)
    return db

def run_rag_pipeline(doc_path="my_notes.txt", gguf_model_path="/content/tinyllama-1.1b-chat.gguf", user_query="What is quantization in LLMs?"):
    prepare_sample_doc(doc_path)
    chunks = load_and_chunk_documents(doc_path)
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = create_or_load_vectorstore(chunks, embedding_model)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
    relevant_docs = retriever.get_relevant_documents(user_query)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    prompt = f"""<|system|>
You are a helpful assistant. Use the following context to answer the user question.

Context:
{context}</s>
<|user|>
{user_query}</s>
<|assistant|>
"""

    print("\n--- Initializing Llama Engine ---")
    llm = Llama(model_path=gguf_model_path, n_ctx=1024, verbose=False)
    print("--- Generating Answer ---")
    output = llm(prompt, max_tokens=150, stop=["</s>"])
    return output["choices"][0]["text"].strip()

answer = run_rag_pipeline()
print("\nResult:")
print(answer)